# Document Intelligence Pipeline: recorrido de punta a punta

Este cuaderno ejecuta el pipeline **sin llave de API** (fallback determinista: pdfplumber + tesseract + reglas) sobre un texto, un PDF con capa de texto, un PDF escaneado y una anotación real de FUNSD, y muestra los campos, hallazgos, routing y telemetría. Con `ANTHROPIC_API_KEY` en `.env` los mismos pasos usan Claude y Claude Vision.

In [1]:
import json, os, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
os.environ.setdefault('LOG_LEVEL', 'WARNING')
from src.config import Settings
from src.observability.tracing import configure_logging
configure_logging('WARNING')
from src.pipeline.graph import DocumentPipeline, build_deps
settings = Settings()
pipeline = DocumentPipeline(build_deps(settings))
print('modelo fijado:', settings.anthropic_model, '| llm_enabled:', settings.llm_enabled)

/home/user/trabajo/document-intelligence-pipeline/.venv/lib/python3.11/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


modelo fijado: claude-sonnet-4-5-20250929 | llm_enabled: False


## 1. Texto de una factura

In [2]:
INVOICE = '''ACME CONSULTING S.A.S.
NIT: 900.123.456-8
Invoice No: INV-2026-0042
Invoice Date: 2026-03-14
Due Date: 2026-04-13
Payment Terms: Net 30
Subtotal: 2,100.00
Tax (19%): 399.00
Total Due: USD 2,499.00'''
result = await pipeline.run(text=INVOICE)
print('tipo:', result.document_type, '| ruta:', result.pipeline_used, '| routing:', result.routing, '| conf:', result.overall_confidence)
for f in result.fields:
    print(f'  {f.name:15s} {str(f.value):22s} conf={f.confidence:.2f} src={f.source}')
print('hallazgos:', [f.code for f in result.findings])

tipo: invoice | ruta: text | routing: human_review | conf: 0.844
  invoice_number  INV-2026-0042          conf=0.88 src=regex
  vendor_tax_id   900.123.456-8          conf=0.92 src=regex
  issue_date      2026-03-14             conf=0.88 src=regex
  due_date        2026-04-13             conf=0.92 src=regex
  subtotal        2100.0                 conf=0.92 src=regex
  tax             399.0                  conf=0.82 src=regex
  total           2499.0                 conf=0.88 src=regex
  payment_terms   Net 30                 conf=0.92 src=regex
  currency        USD                    conf=0.70 src=regex
  vendor          ACME CONSULTING S.A.S. conf=0.60 src=regex
hallazgos: []


## 2. PDF con capa de texto y tabla (pdfplumber)

In [3]:
from src.synth.pdf import render_document_pdf
pdf = render_document_pdf(INVOICE, ['Description', 'Qty', 'Unit price', 'Amount'],
                          [['Consulting hours', '10', '120.00', '1200.00'], ['Cloud hosting', '1', '900.00', '900.00']])
result = await pipeline.run(pdf_bytes=pdf)
print('ruta:', result.pipeline_used, '| páginas:', result.n_pages, '| tablas:', len(result.tables))
if result.tables:
    print(result.tables[0].headers); print(result.tables[0].rows)
print('nodos:', [(n['node'], n['ms']) for n in result.node_log])

ruta: text | páginas: 1 | tablas: 1
['Description', 'Qty', 'Unit price', 'Amount']
[['Consulting hours', '10', '120.00', '1200.00'], ['Cloud hosting', '1', '900.00', '900.00']]
nodos: [('n_ingest', 95), ('n_classify', 0), ('n_tables', 11), ('n_extract', 0), ('n_validate', 0), ('n_score', 0)]


## 3. PDF escaneado (rasterizado, sin capa de texto) → ruta OCR

Si tesseract no está instalado el pipeline lo indica y usa la ruta de texto (vacía).

In [4]:
from src.parsers.ocr_parser import tesseract_available
from src.synth.pdf import render_scanned_pdf
scanned = render_scanned_pdf(pdf)
print('tesseract disponible:', tesseract_available(settings.tesseract_cmd))
result = await pipeline.run(pdf_bytes=scanned, document_type='invoice')
print('ruta:', result.pipeline_used, '| routing:', result.routing, '| latencia ms:', result.latency_ms)
print({f.name: f.value for f in result.fields})

tesseract disponible: True


ruta: ocr | routing: human_review | latencia ms: 519
{'invoice_number': 'INV-2026-0042', 'vendor_tax_id': '900.123.456-8', 'issue_date': '2026-03-14', 'due_date': '2026-04-13', 'subtotal': 2100.0, 'tax': 399.0, 'total': 2499.0, 'payment_terms': 'Net 30', 'currency': 'USD', 'vendor': 'ACME CONSULTING S.A.S.'}


## 4. Formulario real de FUNSD (texto reconstruido de la anotación)

In [5]:
from src.synth.funsd import load_funsd_annotation
doc = load_funsd_annotation(ROOT / 'data/funsd_sample/annotations/0000971160.json')
print(doc.text[:400]); print('...')
print('pares anotados:', doc.pairs[:4])
result = await pipeline.run(text=doc.text)
print('tipo:', result.document_type, '| routing:', result.routing, '| campos:', {f.name: f.value for f in result.fields})

R&D
R&D QUALITY IMPROVEMENT SUGGESTION/ SOLUTION FORM
Name / Phone Ext. :  M. Hamann P. Harper, P. Martinez  Date:  9/ 3/ 92
Supervisor / Manager  :  J. S. Wigand  R&D Group:  Licensee
Suggestion:
Discontinue coal retention analyses on licensee submitted product samples (Note : Coal Retention testing is not performed by most licensees. Other B&W physical measurements as ends stability and inspecti
...
pares anotados: [('Suggestion:', 'Discontinue coal retention analyses on licensee submitted product samples (Note : Coal Retention testing is not performed by most licensees. Other B&W physical measurements as ends stability and inspection for soft spots in ciparettes are thought to be sufficient measures to assure cigarette physical integrity. The proposed action will increase laboratory productivity . )'), ('Date:', '9/ 3/ 92'), ('Name / Phone Ext. :', 'M. Hamann P. Harper, P. Martinez'), ('R&D Group:', 'Licensee')]
tipo: form | routing: vlm_fallback | campos: {'submitter': 'M. Hamann P

## 5. Métricas de la última corrida (`eval/RESULTS.md`)

In [6]:
runs = sorted((ROOT / 'eval/runs').glob('*.json'))
run = json.loads(runs[-1].read_text(encoding='utf-8'))
summary = run['pipelines']['heuristic']['summary']
print(runs[-1].name)
print({k: summary[k] for k in ('documents', 'field_f1_micro', 'field_f1_macro', 'doc_accuracy', 'classifier_accuracy')})
print('tablas cell F1:', run['tables']['f1'], '| latencia:', summary['latency'])
print('pendiente con llave:', [k for k, v in run['pipelines'].items() if v.get('summary') is None])

2026-09-29-heuristic-baseline.json
{'documents': 105, 'field_f1_micro': 0.8289, 'field_f1_macro': 0.8117, 'doc_accuracy': 0.71, 'classifier_accuracy': 0.9905}
tablas cell F1: 0.9672 | latencia: {'mean_ms': 17.8, 'p50_ms': 13.0, 'p95_ms': 41.0}
pendiente con llave: ['fallback', 'vision']
